# A2C ve A3C Teorisi

A2C ve A3C, Actor-Critic ailesinin çoklu ortam/işçi kullanan varyantlarıdır. Temel fikir, daha çeşitli rollout toplayarak politika güncellemelerini daha stabil hale getirmektir.

## 1. A3C: Asynchronous Advantage Actor-Critic

A3C'de birden fazla worker aynı anda kendi ortamlarında rollout toplar. Her worker global modelin kopyasıyla davranır, gradyan hesaplar ve global parametrelere uygular.

Avantajları:

- Farklı worker'lar farklı deneyimler toplar.
- Replay buffer olmadan korelasyon azalır.
- CPU üzerinde paralel çalışmaya uygundur.

Dezavantajı, asenkron gradyanların stale olabilmesidir: worker'ın hesapladığı gradyan, global model değiştikten sonra uygulanabilir.

## 2. A2C: Synchronous Advantage Actor-Critic

A2C on-policy bir yöntemdir. Paralel ortamlar sadece daha çeşitli ve daha hızlı rollout toplamak için kullanılır; toplanan veri yine güncel politikanın verisi olarak kabul edilir. Bu veri genellikle kısa bir update penceresinden sonra yenilenir.

A2C, A3C'nin senkron versiyonudur. Birden fazla ortamdan aynı anda rollout toplanır, sonra tek bir batch olarak güncelleme yapılır.

A2C akışı:

1. $N$ paralel ortamda $T$ adım rollout topla.
2. N-step return veya GAE hesapla.
3. Actor loss, value loss ve entropy loss'u birleştir.
4. Tek optimizer adımı at.

A2C daha deterministik ve debug etmesi daha kolaydır; A3C daha gevşek paralellik sağlar.

## 3. N-Step Return

A2C/A3C çoğunlukla n-step bootstrap kullanır:

$$R_t=r_{t+1}+\gamma r_{t+2}+...+\gamma^{n-1}r_{t+n}+\gamma^nV(s_{t+n})$$

Episode bittiyse bootstrap terimi düşer:

$$R_t=r_{t+1}+\gamma r_{t+2}+...+\gamma^{T-t-1}r_T$$

Advantage:

$$\hat{A}_t=R_t-V(s_t)$$

In [ ]:
import torch

def n_step_returns(rewards, dones, last_value, gamma=0.99):
    returns = torch.zeros_like(rewards)
    running = last_value
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running * (1.0 - dones[t])
        returns[t] = running
    return returns

rewards = torch.tensor([0.0, 0.1, 0.0, 1.0])
dones = torch.tensor([0.0, 0.0, 0.0, 0.0])
n_step_returns(rewards, dones, last_value=torch.tensor(0.5))

## 4.1. A2C İçin Paralel Rollout Ağı

A2C'de ağ klasik Actor-Critic ağıdır; fark, verinin birden fazla ortamdan senkron toplanmasıdır. Tensor şekli genelde `(T, N, ...)` olur:

- `T`: rollout uzunluğu
- `N`: paralel ortam sayısı

Aşağıdaki hücre bu batch yapısını minimal bir ağla gösterir.

In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.distributions import Categorical
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Bu hücre için PyTorch gerekli. Jupyter'da orbit_wars_env kernelini seçin.") from exc

class A2CNet(nn.Module):
    """A2C: actor-critic ağı, paralel rollout batch'inde vektörize çalışır."""
    def __init__(self, obs_dim=8, hidden_dim=64, n_actions=4):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(obs_dim, hidden_dim), nn.ReLU())
        self.actor = nn.Linear(hidden_dim, n_actions)
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, obs):
        h = self.body(obs)
        return self.actor(h), self.critic(h).squeeze(-1)

T, N, obs_dim = 5, 3, 8
net = A2CNet(obs_dim=obs_dim)
rollout_obs = torch.randn(T, N, obs_dim)
flat_obs = rollout_obs.reshape(T * N, obs_dim)
logits, values = net(flat_obs)
logits = logits.reshape(T, N, -1)
values = values.reshape(T, N)

dist = Categorical(logits=logits)
actions = dist.sample()              # (T, N)
log_probs = dist.log_prob(actions)    # (T, N)
advantages = torch.randn(T, N)        # normalde n-step return veya GAE'den gelir
returns = advantages + values.detach()

policy_loss = -(log_probs * advantages.detach()).mean()
value_loss = 0.5 * (values - returns).pow(2).mean()
entropy = dist.entropy().mean()
loss = policy_loss + 0.5 * value_loss - 0.01 * entropy

print("logits:", logits.shape)
print("values:", values.shape)
print("actions:", actions.shape)
print("loss:", float(loss.detach()))

## 4.2. A2C Update Algoritması

A2C update'i Actor-Critic update'inin paralel rollout halidir. `T x N` rollout düzleştirilir, policy/value/entropy loss hesaplanır ve tek senkron optimizer adımı atılır.

In [ ]:
def a2c_update(net, optimizer, rollout_obs, actions, returns, advantages, value_coef=0.5, entropy_coef=0.01):
    T, N, obs_dim = rollout_obs.shape
    flat_obs = rollout_obs.reshape(T * N, obs_dim)
    flat_actions = actions.reshape(T * N)
    flat_returns = returns.reshape(T * N)
    flat_advantages = advantages.reshape(T * N)

    logits, values = net(flat_obs)
    dist = Categorical(logits=logits)
    log_probs = dist.log_prob(flat_actions)

    policy_loss = -(log_probs * flat_advantages.detach()).mean()
    value_loss = 0.5 * (values - flat_returns).pow(2).mean()
    entropy = dist.entropy().mean()
    loss = policy_loss + value_coef * value_loss - entropy_coef * entropy

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=0.5)
    optimizer.step()

    return {
        "loss": float(loss.detach()),
        "policy_loss": float(policy_loss.detach()),
        "value_loss": float(value_loss.detach()),
        "entropy": float(entropy.detach()),
    }

net = A2CNet(obs_dim=8, n_actions=4)
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
T, N, obs_dim = 5, 3, 8
rollout_obs = torch.randn(T, N, obs_dim)
with torch.no_grad():
    logits, values = net(rollout_obs.reshape(T * N, obs_dim))
    actions = Categorical(logits=logits.reshape(T, N, -1)).sample()
returns = torch.randn(T, N)
advantages = returns - values.reshape(T, N)
a2c_update(net, optimizer, rollout_obs, actions, returns, advantages)

## 4. Kayıp Fonksiyonu

A2C/A3C için tipik toplam kayıp:

$$L=-\log\pi_\theta(a_t\mid s_t)\hat{A}_t+c_v(R_t-V_\phi(s_t))^2-c_eH(\pi_\theta)$$

Burada:

- İlk terim actor'ı günceller.
- İkinci terim critic'i eğitir.
- Entropy terimi keşfi destekler.

## 5. PPO ile İlişkisi

PPO, A2C'ye benzer rollout toplar fakat policy update'i clipped ratio ile sınırlar:

$$r_t(\theta)=\frac{\pi_\theta(a_t\mid s_t)}{\pi_{old}(a_t\mid s_t)}$$

A2C'de böyle bir ratio clipping yoktur. Bu yüzden A2C daha basit ama büyük update adımlarına daha hassastır. PPO genellikle aynı actor-critic altyapısının daha stabil versiyonu olarak düşünülebilir.

## 6. Orbit Wars İçin Yorum

Orbit Wars self-play olduğu için rollout dağılımı sürekli değişir. A2C kullanılacaksa:

- Birden fazla seed/oyun paralel toplanmalı.
- Advantage normalize edilmeli.
- Entropy dikkatli ayarlanmalı.
- Update başına rollout sayısı küçükse gradyan çok noisy olabilir.

Mevcut repo PPO kullandığı için A2C'nin temel actor-critic mantığı zaten var; PPO sadece update'i daha kontrollü hale getiriyor.

## 7. A2C ve A3C Karşılaştırması

| Özellik | A2C | A3C |
| --- | --- | --- |
| Güncelleme | Senkron | Asenkron |
| Debug | Daha kolay | Daha zor |
| Donanım | GPU batch'e uygun | CPU worker'lara uygun |
| Gradyan | Tek batch'ten gelir | Worker bazlı ve stale olabilir |
| Pratik kullanım | Modern kodlarda daha yaygın | Tarihsel olarak önemli |